In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
# schema_cbr.py (case based reasoning)
from pydantic import BaseModel, Field
from typing import List, Optional

class Concept(BaseModel):
    name: str = Field(
        description="Broad mathematical concept (e.g., 'Complex Analysis', 'Probability')."
    )

class TheoremFormula(BaseModel):
    name: str = Field(
        description="Specific theorem, rule, or formula (e.g., 'Erdos-Turan Theorem', 'Residue Theorem')."
    )
    latex_expression: str = Field(
        description="The core LaTeX representation. Leave empty if none."
    )

class CanonicalProblem(BaseModel):
    """
    This is the core of our Case-Based Reasoning. 
    It distills a messy Q&A into a strict, reusable mathematical case.
    """
    objective: str = Field(
        description="A clear, one-sentence statement of the mathematical goal."
    )
    math_setup: str = Field(
        description="The initial conditions, variables, and constraints, heavily utilizing LaTeX."
    )
    solution_steps: List[str] = Field(
        description="An array of logical steps to solve the problem, referencing the theorems used."
    )

class MathCaseExtraction(BaseModel):
    """
    The final object returned by the LLM containing the distilled case and its entities.
    """
    canonical_problem: CanonicalProblem = Field(
        description="The clean, distilled version of the raw Question and Answer."
    )
    concepts: List[Concept] = Field(
        description="Mathematical concepts required to understand this case."
    )
    theorems: List[TheoremFormula] = Field(
        description="Specific theorems and formulas applied in the solution steps."
    )
    dependencies: List[dict] = Field(
        default_factory=list,
        description="Optional: List of dicts with 'source' and 'target' showing if a theorem depends on a concept."
    )

# graph topology
# node: [Canonical_Problem] (ID: canonical_182412)

# node: [Concept] (ID: concept_polynomials)

# node: [Theorem] (ID: theorem_erdos_turan)

# edges:

#     (canonical_182412) --[BELONGS_TO_CONCEPT]--> (concept_polynomials)

#     (canonical_182412) --[SOLVED_WITH_THEOREM]--> (theorem_erdos_turan)

In [10]:
# extractor_cbr.py
import json
import os
from groq import Groq
from pydantic import ValidationError

from dotenv import load_dotenv
load_dotenv()

# from schema_cbr import MathCaseExtraction
from math_assistant_agent.agent import get_llm_client, llm_response

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = get_llm_client(provider_name="groq", api_key=GROQ_API_KEY)

In [12]:
# MathCaseExtraction.model_json_schema()

In [ ]:
def extract_canonical_case(
    client, 
    provider_name: str,
    question_title: str, 
    question_text: str, 
    answer_text: str,
    model: str = "openai/gpt-oss-20b",
    temperature: float = 0.1
) -> MathCaseExtraction:
    """
    Reads a messy raw Q&A pair and distills it into a clean, canonical mathematical case.
    """
    print(f"Distilling raw data into Canonical Case: '{question_title[:40]}...'")
    
    # Dynamically generate the JSON schema from our Pydantic model
    # This ensures the LLM always sees the most up-to-date schema structure
    json_schema = MathKnowledgeExtraction.model_json_schema()
    
    system_prompt = f"""
    You are an expert mathematician and ontologist.
    Your task is to analyze a mathematical Question and its Accepted Answer.
    Extract the core concepts, specific theorems/formulas used, and the logical dependencies between them.
    
    You MUST return ONLY a valid JSON object that strictly adheres to the following schema:
    {json.dumps(json_schema, indent=2)}
    
    CRITICAL INSTRUCTIONS:
    1. Ensure 'latex_expression' contains proper LaTeX code without the enclosing $ or $$. Leave empty if not applicable.
    2. 'source_entity' and 'target_entity' in dependencies MUST match the names of the concepts or theorems you extracted.
    """
    
    question = f"""
    QUESTION TITLE: {question_title}
    QUESTION BODY: {question_text}
    ACCEPTED ANSWER: {answer_text}
    """
    
    try:
        raw_json_str = llm_response(
            client=client, 
            provider_name=provider_name, 
            system_prompt=system_prompt, 
            question=question, 
            model=model, 
            temperature=temperature, 
            json_mode=True
        )
        
        parsed_dict = json.loads(raw_json_str)
        
        # Double Validation: Pass the raw dictionary back into Pydantic
        # This will raise a ValidationError if the LLM hallucinated the wrong structure
        validated_knowledge = MathKnowledgeExtraction(**parsed_dict)
        
        print("Extraction successful and validated!")
        return validated_knowledge

    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON from LLM: {e}")
        return None
    except ValidationError as e:
        print(f"LLM output did not match Pydantic schema: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [8]:
# fast test
if __name__ == "__main__":
    from dotenv import load_dotenv
    load_dotenv()
    
    # Mock raw data
    MOCK_TITLE = "Why do roots of polynomials tend to have absolute value close to 1?"
    MOCK_Q = "While playing around with Mathematica I noticed that most polynomials with real coefficients seem to have zeroes near the unit circle..."
    MOCK_A = "This is a consequence of the Erdos-Turan theorem. When coefficients are chosen from bounded intervals, the roots asymptotically cluster around the unit circle. Let P(z) be our polynomial..."
    
    extracted_data = extract_math_knowledge(
        client=client,
        provider_name="groq",
        question_title=MOCK_TITLE,
        question_text=MOCK_Q,
        answer_text=MOCK_A
    )
    
    if extracted_data:
        print("\n--- VALIDATED EXTRACTION ---")
        # .model_dump_json() is a Pydantic method to neatly print the object
        print(extracted_data.model_dump_json(indent=2))

Extracting semantic knowledge from: 'Why do roots of polynomials tend to have...'
Extraction successful and validated!

--- VALIDATED EXTRACTION ---
{
  "concepts": [
    {
      "name": "Polynomials with real coefficients"
    },
    {
      "name": "Roots of polynomials"
    },
    {
      "name": "Unit circle"
    },
    {
      "name": "Bounded coefficients"
    },
    {
      "name": "Asymptotic behavior"
    },
    {
      "name": "Erdos-Turan theorem"
    }
  ],
  "theorems_formulas": [
    {
      "name": "Erdos-Turan theorem",
      "latex_expression": "P(z)=a_n z^n + \\dots + a_0"
    }
  ],
  "dependencies": [
    {
      "source_entity": "Erdos-Turan theorem",
      "target_entity": "Bounded coefficients",
      "relationship_type": "DEPENDS_ON"
    },
    {
      "source_entity": "Erdos-Turan theorem",
      "target_entity": "Unit circle",
      "relationship_type": "DEPENDS_ON"
    },
    {
      "source_entity": "Roots of polynomials",
      "target_entity": "Erdos-Turan

In [15]:
# graph_builder_cot.py
import re 
from typing import Dict, Any 
# from schema_cot import MathKnowledgeExtraction
from math_assistant_agent.data import load_graph_json

GRAPH_DATA = load_graph_json(
    "/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-19-21.json"
)

In [16]:
GRAPH_DATA

{'nodes': [{'id': 'question_182412',
   'label': 'Question',
   'properties': {'question_id': 182412,
    'title': 'Why do roots of polynomials tend to have absolute value close to 1?',
    'text': 'While playing around with Mathematica I noticed that most polynomials with real coefficients seem to have most complex zeroes very near the unit circle. For instance, if we plot all the roots of a polynomial of degree 300 with coefficients chosen randomly from the interval $[27, 42]$, we get something like this:\n\nThe Mathematica code to produce the picture was:\n\nrandomPoly[n_, x_, {a_, b_}] := \n  x^Range[0, n] . Table[RandomReal[{a, b}], {n + 1}];\nGraphics[Point[{Re[x], Im[x]}] /. \n  NSolve[randomPoly[300, x, {27, 42}], x], Axes -> True]\n\nIf I try other intervals and other degrees, the picture is always mostly the same: almost all roots are close to the unit circle.\n\nQuestion: why does this happen?',
    'score': 446,
    'view_count': 69367,
    'link': 'https://mathoverflow.net

In [17]:
def create_friendly_id(prefix: str, text: str) -> str:
    """
    Converts a string like 'Erdos-Turan Theorem' into a safe ID like 'theorem_erdos_turan_theorem'.
    """
    clean_text = re.sub(r'[^a-z0-9]+', '_', text.lower().strip())
    return f"{prefix}_{clean_text}"

def build_semantic_graph(
    graph_data: Dict[str, list], 
    question_id: str, 
    answer_id: str, 
    extracted_knowledge: MathKnowledgeExtraction
) -> Dict[str, list]:
    """
    Injects the extracted semantic knowledge (Concepts, Theorems, Dependencies) 
    into the main graph structure.
    """
    print(f"Building graph connections for Question: {question_id}")

    # Helper function to check if a node already exists
    def node_exists(node_id: str) -> bool:
        return any(n['id'] == node_id for n in graph_data['nodes'])

    # Helper function to check if an edge already exists
    def edge_exists(source: str, target: str, rel_type: str) -> bool:
        return any(
            e['source'] == source and e['target'] == target and e['type'] == rel_type 
            for e in graph_data['edges']
        )

# 1. Process Concepts (Logic remains the same, which is good)
    for concept in extracted_knowledge.concepts:
        concept_id = create_friendly_id("concept", concept.name)
        if not node_exists(concept_id):
            graph_data["nodes"].append({
                "id": concept_id,
                "label": "Concept",
                "properties": {"name": concept.name}
            })
        if not edge_exists(question_id, concept_id, "ADDRESSES_CONCEPT"):
            graph_data["edges"].append({
                "source": question_id,
                "target": concept_id,
                "type": "ADDRESSES_CONCEPT"
            })

    # 2. Process Theorems
    for theorem in extracted_knowledge.theorems_formulas:
        theorem_id = create_friendly_id("theorem", theorem.name)
        if not node_exists(theorem_id):
            graph_data["nodes"].append({
                "id": theorem_id,
                "label": "Theorem",
                "properties": {"name": theorem.name, "latex": theorem.latex_expression}
            })
        if not edge_exists(answer_id, theorem_id, "USES_THEOREM"):
            graph_data["edges"].append({
                "source": answer_id,
                "target": theorem_id,
                "type": "USES_THEOREM"
            })

    # 3. Process Dependencies (Improved logic)
    for dep in extracted_knowledge.dependencies:
        # Determine IDs based on normalized names
        # Note: We look for the node in both categories because the LLM 
        # might be ambiguous about whether something is a Concept or a Theorem
        src_id = next((n['id'] for n in graph_data['nodes'] 
                       if n['id'] in [create_friendly_id("concept", dep.source_entity), 
                                      create_friendly_id("theorem", dep.source_entity)]), None)
        
        tgt_id = next((n['id'] for n in graph_data['nodes'] 
                       if n['id'] in [create_friendly_id("concept", dep.target_entity), 
                                      create_friendly_id("theorem", dep.target_entity)]), None)

        if src_id and tgt_id:
            rel_type = dep.relationship_type.upper().replace(" ", "_")
            if not edge_exists(src_id, tgt_id, rel_type):
                graph_data["edges"].append({
                    "source": src_id,
                    "target": tgt_id,
                    "type": rel_type
                })
        else:
            print(f"Skipping dependency: {dep.source_entity} -> {dep.target_entity} (Nodes not found)")

    return graph_data

In [25]:
import time
import json
# from extractor_cot import get_groq_client, extract_math_knowledge
# from graph_builder_cot import build_semantic_graph

def run_batch_extraction(client, raw_graph_data):
    
    # 1. Identificar pares de (Question, Answer) no seu grafo
    # Criamos um dicionário para busca rápida de respostas pelas perguntas
    questions = [n for n in raw_graph_data["nodes"] if n["label"] == "Question"]
    
    print(f"Starting batch extraction for {len(questions)} questions...")

    for i, q_node in enumerate(questions):
        q_id = q_node["id"]
        
        # Encontrar a resposta associada
        edge = next((e for e in raw_graph_data["edges"] 
                     if e["source"] == q_id and e["type"] == "HAS_ACCEPTED_ANSWER"), None)
        
        if not edge:
            continue
            
        a_id = edge["target"]
        a_node = next((n for n in raw_graph_data["nodes"] if n["id"] == a_id), None)
        
        if not a_node:
            continue

        # 2. Chamada Real ao LLM
        print(f"[{i+1}/{len(questions)}] Processing: {q_node['properties']['title'][:50]}")
        
        knowledge = extract_math_knowledge(
            client=client,
            provider_name="groq", 
            question_title=q_node["properties"].get("title", ""),
            question_text=q_node["properties"].get("text", ""),
            answer_text=a_node["properties"].get("text", "")
        )

        # 3. Construção Semântica no Grafo
        if knowledge:
            raw_graph_data = build_semantic_graph(
                graph_data=raw_graph_data,
                question_id=q_id,
                answer_id=a_id,
                extracted_knowledge=knowledge
            )
        
        # 4. Respeitar o Rate Limit do Groq
        # O modelo GPT-OSS-20b tem 8K TPM. 
        # Cada requisição consome aprox 1.5K-2K tokens.
        # Esperar 15s garante uma margem de segurança.
        print("Waiting 10 to respect rate limits...")
        time.sleep(2)

    return raw_graph_data

In [26]:
# ==========================================
# EXECUÇÃO
# ==========================================
if __name__ == "__main__":
    
    final_graph = run_batch_extraction(client=client, raw_graph_data=GRAPH_DATA)
    
    # Salvar o novo grafo enriquecido
    with open("outputs/semantic_graph_cot.json", "w") as f:
        json.dump(final_graph, f, indent=2)
    print("Processamento concluído! Grafo semântico salvo.")

Starting batch extraction for 5 questions...
[1/5] Processing: Why do roots of polynomials tend to have absolute 
Extracting semantic knowledge from: 'Why do roots of polynomials tend to have...'
Extraction successful and validated!
Building graph connections for Question: question_182412
Waiting 10 to respect rate limits...
[2/5] Processing: Philosophy behind Mochizuki&#39;s work on the ABC 
Extracting semantic knowledge from: 'Philosophy behind Mochizuki&#39;s work o...'
Extraction successful and validated!
Building graph connections for Question: question_106560
Waiting 10 to respect rate limits...
[3/5] Processing: Which journals publish expository work?
Extracting semantic knowledge from: 'Which journals publish expository work?...'
An unexpected error occurred: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxsa2evmf07948nc17h47ss2` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 3871, Reques

In [3]:
from math_assistant_agent.visualization.pyvis_renderer_v2 import render_graph
from math_assistant_agent.data import load_graph_json

In [5]:
GRAPH_DATA = load_graph_json("outputs/semantic_graph_cot.json")
render_graph(
    graph_data=GRAPH_DATA, 
    output_path="outputs/semantic_graph_cot.html"
)

Rendering visualization for 74 nodes and 90 edges...
✅ Graph rendered successfully! Open 'outputs/semantic_graph_cot.html' in your browser.
